# Fine-tune Qwen3 for n8n workflow diagnosis and repair

Train the n8n workflow repair recipe on a small Apache-2.0 Qwen3 text model. Every training step is visible below. You will finish with a LoRA adapter, a Q4_K_M GGUF, and a ZIP containing actual held-out diagnosis and repair outputs.

Choose **Runtime > Change runtime type > T4 GPU**, then save a copy to Drive. Run the cells in order. You do not need a Hugging Face token.

The optional own-data section is last and is off by default.

## 1. Check the GPU

Confirm the runtime reports an NVIDIA GPU before installing the trainer.

In [ ]:
!nvidia-smi

## 2. Install Unsloth

This follows [Unsloth’s Qwen3 Colab](https://github.com/unslothai/notebooks/blob/main/nb/Qwen3_%2814B%29-Reasoning-Conversational.ipynb) installation procedure, checked 21 November 2026. Install logs stay visible. If Colab requests a runtime restart, restart it, then continue with Settings. Source notebooks describe the installation procedure; this tutorial and its data are original.

In [ ]:
# Follow Unsloth's official Qwen3 Colab installation sequence.
%pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer pyyaml
%pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
!unsloth install-kernels
%pip install --no-deps --upgrade "torchao>=0.16.0"
%pip install transformers==4.57.6
%pip install --no-deps trl==0.22.2

## 3. Settings and recipe

Choose the model alias and recipe here. Each execution creates a separate output directory.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, os, json, hashlib, time, platform, importlib.metadata
import yaml

REPO_URL = "https://github.com/ProMBZ/finetune-anywhere.git"
REPO_BRANCH = "project-01-n8n-workflow-repair"
ROOT = Path("/content/finetune-anywhere")

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(ROOT)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

os.chdir(ROOT)
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))


RECIPE = "n8n-workflow-repair"
MODEL_ALIAS = "starter"  # Qwen3-1.7B from the starter model registry.
DATA_DIR = ROOT / 'recipes' / RECIPE / 'data'
config = yaml.safe_load((ROOT / 'recipes' / RECIPE / 'config.yaml').read_text())
registry = yaml.safe_load((ROOT / 'configs/models.yaml').read_text())
model_entry = registry['models'][MODEL_ALIAS]
RUN_NAME = RECIPE + '-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUTPUT = ROOT.parent / 'finetune-runs' / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=False)
print('Model:', model_entry['id'])
print('Data:', DATA_DIR)
print('Output:', OUTPUT)
print('Repository revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## 4. Optional Drive backup

Enable this before training to keep checkpoints in your Drive. Leave it false to use session storage and download the results at the end.

In [ ]:
SAVE_TO_DRIVE = True
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/finetune-anywhere') / RUN_NAME
    OUTPUT.mkdir(parents=True, exist_ok=False)
print(OUTPUT)

## 5. Load the model

Load the base in four-bit precision to leave room for training activations on the T4.

In [ ]:
# Unsloth must be imported before Transformers, PEFT or TRL.
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

assert torch.cuda.is_available(), 'Select a GPU runtime, then restart from the first cell.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_entry['id'],  # The verified model ID comes from the registry.
    max_seq_length=config['max_seq_length'],  # Limit conversation length to fit memory.
    dtype=None,  # Let Unsloth select the hardware-appropriate weight dtype.
    load_in_4bit=True,  # Keep the frozen base weights in 4 bit for QLoRA.
)

## 6. Apply LoRA

Train small adapter matrices while keeping the base frozen. Each setting is explained beside its value.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=config['rank'],  # Rank 16 controls how much the adapter can learn.
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],  # Adapt attention and feed-forward projections.
    lora_alpha=config['alpha'],  # Alpha 32 scales the adapter update relative to its rank.
    lora_dropout=0,  # Zero dropout uses Unsloth's optimized LoRA path.
    bias='none',  # Leave the original bias parameters frozen.
    use_gradient_checkpointing='unsloth',  # Recompute activations to reduce training memory.
    random_state=config['seed'],  # Fix the adapter initialization seed.
    use_rslora=False,  # Use ordinary LoRA scaling for this starter.
    loftq_config=None,  # Use standard adapter initialization without LoftQ.
)
model.print_trainable_parameters()

## 7. Set the chat template

Use Qwen3’s own role delimiters and an empty thinking prefix for direct answers. The saved tokenizer carries this setting into export.

In [ ]:
# Persist non-thinking mode so training, reloads and GGUF share the same template.
prefix = '{% set enable_thinking = false %}\n'
assert isinstance(tokenizer.chat_template, str) and 'enable_thinking' in tokenizer.chat_template
if not tokenizer.chat_template.startswith(prefix):
    tokenizer.chat_template = prefix + tokenizer.chat_template

preview = [{'role': 'user', 'content': 'Please order 3 desk lamps.'},
           {'role': 'assistant', 'content': '{"item":"desk lamps","quantity":3,"priority":"normal"}'}]
print(tokenizer.apply_chat_template(preview, tokenize=False, add_generation_prompt=False, enable_thinking=False))

## 8. Load and check the data

Validate exact token lengths and split separation before training. Render chat text once, without adding special tokens twice.

In [ ]:
from datasets import Dataset
from common import read_rows
from validate_dataset import validate
from validate_n8n_dataset import validate_dataset as validate_n8n_recipe

recipe_validation = validate_n8n_recipe(DATA_DIR)
print('Recipe structural validation:', recipe_validation)

raw = {}
seen_prompts = set()
for split in ['train', 'valid', 'eval']:
    file = DATA_DIR / f'{split}.jsonl'
    errors, lengths, prompts = validate(file, tokenizer=tokenizer, max_tokens=config['max_seq_length'])
    if errors:
        raise ValueError('\n'.join(errors))
    if seen_prompts & prompts:
        raise ValueError(f'{split}: a prompt appears in another split')
    seen_prompts |= prompts
    raw[split] = read_rows(file)
    print(split, len(raw[split]), 'examples; longest:', max(lengths), 'tokens')

def render_chat(example):
    return {'text': tokenizer.apply_chat_template(example['messages'], tokenize=False,
                    add_generation_prompt=False, enable_thinking=False)}

train_data = Dataset.from_list(raw['train']).map(render_chat, remove_columns=['messages'])
valid_data = Dataset.from_list(raw['valid']).map(render_chat, remove_columns=['messages'])
# raw['eval'] stays outside the trainer. Its reference answers are never input to generation.

# Build a small local BM25 index from the official n8n Markdown documentation.
# Retrieved excerpts are supplied to BOTH base and fine-tuned evaluation prompts.
from n8n_docs_retriever import build_index, augment_messages

DOCS_INDEX, DOCS_REPORT = build_index(
    ROOT / 'recipes' / RECIPE / 'knowledge' / 'sources.yaml',
    OUTPUT / 'knowledge',
)
print('Official docs indexed:', DOCS_REPORT['sources_indexed'], '/', DOCS_REPORT['sources_requested'])
print('Documentation chunks:', DOCS_REPORT['chunks'])
if DOCS_REPORT['sources_indexed'] == 0:
    raise RuntimeError('No official n8n documentation sources could be indexed. Check the Colab network/logs.')

eval_prompts = [
    augment_messages(row['messages'][:-1], DOCS_INDEX, top_k=2, excerpt_chars=240)
    for row in raw['eval']
]

def prompt_token_length(messages):
    encoded = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, enable_thinking=False
    )
    return len(encoded)

eval_prompt_lengths = [prompt_token_length(messages) for messages in eval_prompts]
if max(eval_prompt_lengths) > config['max_seq_length']:
    eval_prompts = [
        augment_messages(row['messages'][:-1], DOCS_INDEX, top_k=1, excerpt_chars=120)
        for row in raw['eval']
    ]
    eval_prompt_lengths = [prompt_token_length(messages) for messages in eval_prompts]
if max(eval_prompt_lengths) > config['max_seq_length']:
    raise ValueError(
        f'Retrieved evaluation prompt too long: {max(eval_prompt_lengths)} tokens; '
        f"limit is {config['max_seq_length']}."
    )
print('Retrieved evaluation prompts:', len(eval_prompts))
print('Longest retrieved prompt:', max(eval_prompt_lengths), 'tokens')
print('Documentation failures:', DOCS_REPORT['failed_sources'])
print(train_data[0]['text'])

## 9. Record the base answers

Disable the adapter and run all held-out prompts before training. Keep the final assistant targets out of the input.

In [ ]:
GENERATION = {'max_new_tokens': 256, 'do_sample': False}

def answer(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False,
        add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        generated = model.generate(**inputs, **GENERATION, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
base_outputs = []
with model.disable_adapter():
    for messages in eval_prompts:
        base_outputs.append(answer(messages))
print('First held-out prompt:', eval_prompts[0][-1]['content'])
print('BASE:', base_outputs[0])
FastLanguageModel.for_training(model)

## 10. Train

Train on the training split and inspect validation loss after each epoch. The final evaluation split is never passed to the trainer.

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_data,
    eval_dataset=valid_data,
    args=SFTConfig(
        output_dir=str(OUTPUT / 'checkpoints'),  # Keep this run's checkpoints together.
        dataset_text_field='text',  # Train on the rendered conversations above.
        max_length=config['max_seq_length'],  # Use the same length checked during validation.
        per_device_train_batch_size=config['batch_size'],  # One conversation per microbatch.
        gradient_accumulation_steps=config['gradient_accumulation_steps'],  # Accumulate four microbatches per update.
        learning_rate=config['learning_rate'],  # Start with a 0.0001 adapter learning rate.
        num_train_epochs=config['epochs'],  # Make three passes through the 200 training examples.
        warmup_ratio=0.1,  # Increase the learning rate gradually at the start.
        optim='adamw_8bit',  # Store optimizer state in 8 bit to save memory.
        fp16=not is_bfloat16_supported(),  # Use FP16 on the T4.
        bf16=is_bfloat16_supported(),  # Use BF16 on hardware that supports it.
        per_device_eval_batch_size=1,  # Keep validation memory small too.
        eval_strategy='epoch',  # Measure validation loss after each pass.
        save_strategy='epoch',  # Save a checkpoint at the same interval.
        save_total_limit=2,  # Retain the last two checkpoints.
        logging_steps=1,  # Show every optimizer update in the cell output.
        seed=config['seed'],  # Keep the shuffle seed fixed.
        report_to='none',  # Keep logs local to this notebook.
        packing=False,  # Keep each conversation separate.
        padding_free=False,  # Use ordinary padded batches on the T4.
    ),
)
FastLanguageModel.for_training(model)
torch.cuda.reset_peak_memory_stats()
training_start = time.monotonic()
training_result = trainer.train()
training_seconds = time.monotonic() - training_start
print('Training seconds:', round(training_seconds, 2))
print('Peak allocated VRAM (GiB):', round(torch.cuda.max_memory_allocated() / 2**30, 2))

## 11. Test one prompt

Generate the same held-out prompt with the trained adapter and print it beside the actual base answer.

In [ ]:
FastLanguageModel.for_inference(model)
test_messages = eval_prompts[0]
first_tuned_output = answer(test_messages)
print('PROMPT:', test_messages[-1]['content'])
print('BASE:', base_outputs[0])
print('FINE-TUNED:', first_tuned_output)
print('REFERENCE:', raw['eval'][0]['messages'][-1]['content'])

## 12. Evaluate all held-out prompts

Keep every result, including failures. The default recipe also reports exact parsed-JSON matches.

In [ ]:
from IPython.display import display, Markdown
from common import cell

tuned_outputs = [first_tuned_output] + [answer(messages) for messages in eval_prompts[1:]]
records = [{'messages': messages, 'reference': row['messages'][-1]['content'],
            'base_output': before, 'finetuned_output': after}
           for row, messages, before, after in zip(raw['eval'], eval_prompts, base_outputs, tuned_outputs)]
lines = ['# Held-out comparison', '',
         'Same retrieved official-doc context, same 4-bit base, same prompts and generation settings. Comparison is before GGUF export.', '',
         '| Prompt | Reference | Base | Fine-tuned |', '| --- | --- | --- | --- |']
for row in records:
    lines.append('| ' + ' | '.join(cell(value) for value in [row['messages'][-1]['content'],
                 row['reference'], row['base_output'], row['finetuned_output']]) + ' |')
comparison_md = '\n'.join(lines) + '\n'
display(Markdown(comparison_md))

if RECIPE == 'json-extractor':
    def matches(output, reference):
        try:
            parsed = json.loads(output)
            return (isinstance(parsed, dict) and set(parsed) == {'item', 'quantity', 'priority'}
                    and (parsed['item'] is None or isinstance(parsed['item'], str))
                    and (parsed['quantity'] is None or type(parsed['quantity']) is int)
                    and parsed == json.loads(reference))
        except (ValueError, TypeError):
            return False
    for name in ['base_output', 'finetuned_output']:
        print(name, 'exact JSON matches:', sum(matches(row[name], row['reference']) for row in records), '/', len(records))

## 13. Score n8n repair quality

Score the base and fine-tuned model on the same untouched evaluation cases. This checks JSON structure, diagnosis labels, expected patch match, safe abstention and whether the patch can be applied to the supplied synthetic workflow fixture. Patch applicability is **not** proof of execution in a live n8n instance.

In [ ]:
from score_n8n_repairs import score_outputs
from IPython.display import display, Markdown

references = [row['messages'][-1]['content'] for row in raw['eval']]
prompts = [messages[-1]['content'] for messages in eval_prompts]
base_scores = score_outputs(base_outputs, references, prompts)
tuned_scores = score_outputs(tuned_outputs, references, prompts)

def show_scores(label, scores):
    rows = [f"| {key.replace('_', ' ').title()} | {value} |" for key, value in scores.items()
            if isinstance(value, str) and key != 'interpretation']
    display(Markdown(f"### {label}\n\n| Metric | Result |\n|---|---:|\n" + "\n".join(rows)))
    print(scores['interpretation'])

show_scores('Base Qwen3-1.7B', base_scores)
show_scores('Fine-tuned Qwen3-1.7B', tuned_scores)
print('Per-category fine-tuned results:', tuned_scores['by_category'])

## 14. Save adapter and evidence
,
,Save before export so the training evidence is already on disk.

In [ ]:
# Keep adapters and evidence before attempting the larger GGUF export.
model.save_pretrained(str(OUTPUT / 'adapter'))
tokenizer.save_pretrained(str(OUTPUT / 'adapter'))

report = {
    'recipe': RECIPE, 'model_alias': MODEL_ALIAS, 'model_id': model_entry['id'],
    'loaded_base_revision': getattr(model.config, '_commit_hash', None),
    'catalog_revision': model_entry['revision'],
    'repository_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'gpu': torch.cuda.get_device_name(0), 'gpu_bytes': torch.cuda.get_device_properties(0).total_memory,
    'python': platform.python_version(),
    'package_versions': {name: importlib.metadata.version(name) for name in ['unsloth','unsloth_zoo','torch','transformers','trl','peft','datasets']},
    'settings': config, 'enable_thinking': False, 'generation': GENERATION,
    'retrieval_enabled': True, 'documentation_index': DOCS_REPORT,
    'training_seconds': training_seconds, 'train_metrics': training_result.metrics,
    'peak_allocated_gpu_bytes': torch.cuda.max_memory_allocated(),
    'data_sha256': {split: hashlib.sha256((DATA_DIR / f'{split}.jsonl').read_bytes()).hexdigest() for split in raw},
    'split_sizes': {split: len(rows) for split, rows in raw.items()},
    'export_completed': False,
}
(OUTPUT / 'run.json').write_text(json.dumps(report, indent=2) + '\n')
(OUTPUT / 'comparison.json').write_text(json.dumps(records, indent=2, ensure_ascii=False) + '\n')
(OUTPUT / 'comparison.md').write_text(comparison_md)
(OUTPUT / 'training-log.json').write_text(json.dumps(trainer.state.log_history, indent=2) + '\n')
print('Saved adapter and evaluation evidence:', OUTPUT)

## 15. Export GGUF
,
,Merge the adapter and quantize it to Q4_K_M for Ollama or llama.cpp. Keep the cell output if conversion fails.

In [ ]:
# This merges the adapter and writes a portable Q4_K_M GGUF.
export_start = time.monotonic()
model.save_pretrained_gguf(str(OUTPUT / 'gguf'), tokenizer, quantization_method='q4_k_m')
# Unsloth may append _gguf to the requested directory (gguf -> gguf_gguf).
gguf_files = sorted(path for path in OUTPUT.rglob('*.gguf')
                    if path.is_file() and path.stat().st_size > 0
                    and 'Q4_K_M' in path.name.upper())
assert gguf_files, f'No nonempty Q4_K_M GGUF found under {OUTPUT}. Check the export log for its destination.'
report['export_completed'] = True
report['export_seconds'] = time.monotonic() - export_start
report['gguf_files'] = [{'name': f.name, 'path': str(f.relative_to(OUTPUT)), 'bytes': f.stat().st_size} for f in gguf_files]
(OUTPUT / 'run.json').write_text(json.dumps(report, indent=2) + '\n')
for file in gguf_files:
    print(file, round(file.stat().st_size / 1e9, 2), 'GB')

## 16. Download evidence and model
,
,Run this even if the export cell failed. Send the small evidence ZIP back; keep the model files for local inference.

In [ ]:
import zipfile
bundle = OUTPUT / 'run-evidence.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in ['run.json', 'comparison.json', 'comparison.md', 'training-log.json']:
        archive.write(OUTPUT / name, arcname=name)
    docs_manifest = OUTPUT / 'knowledge' / 'manifest.json'
    if docs_manifest.exists():
        archive.write(docs_manifest, arcname='knowledge/manifest.json')
print('Send this small file back for real before/after examples:', bundle)
print('Keep your GGUF and adapter separately; weights are not included in the evidence ZIP.')
from google.colab import files
files.download(str(bundle))
DOWNLOAD_GGUF = False
if DOWNLOAD_GGUF:
    downloads = sorted(path for path in OUTPUT.rglob('*.gguf')
                       if path.is_file() and path.stat().st_size > 0
                       and 'Q4_K_M' in path.name.upper())
    assert downloads, f'No nonempty Q4_K_M GGUF found under {OUTPUT}.'
    for file in downloads:
        files.download(str(file))

For local inference, follow [Run locally](https://github.com/junaiddshaukat/finetune-anywhere/blob/main/guides/06-run-locally.md). The evidence ZIP records pre-export adapter results. Compare the exported GGUF separately on your laptop.

## 17. Train on your own data
,
,Prepare three chat JSONL files with distinct prompts. Enable the cell below, then restart the session and reload a fresh base model using the printed settings. Choose your own learning settings and system prompt for the new task.

In [ ]:
# Optional final cell. Run only when you want to start a NEW experiment with your own data.
USE_OWN_DATA = False
if USE_OWN_DATA:
    from google.colab import files
    uploaded = files.upload()  # Select train.jsonl, valid.jsonl and eval.jsonl together.
    required = {'train.jsonl', 'valid.jsonl', 'eval.jsonl'}
    if set(uploaded) != required:
        raise ValueError('Upload exactly train.jsonl, valid.jsonl and eval.jsonl.')
    own_dir = ROOT.parent / 'my-chat-data'
    own_dir.mkdir(exist_ok=True)
    for name, data in uploaded.items():
        (own_dir / name).write_bytes(data)
    subprocess.run([sys.executable, 'scripts/validate_dataset.py',
        *[str(own_dir / name) for name in sorted(required)], '--tokenizer', MODEL_ALIAS,
        '--max-tokens', str(config['max_seq_length'])], check=True)
    print('Restart the session, rerun Install and Settings, then set:')
    print(f'DATA_DIR = Path({str(own_dir)!r})')
    print("RECIPE = 'my-task'")
    print("config['recipe'] = RECIPE")
    print('Apply those lines in a new cell after Settings, then run Load model through Download evidence.')